# labels: shared label and square conventions

Every other notebook pulls these in with `%run ./labels.ipynb`.

* Squares use python-chess indexing: a1 = 0, b1 = 1, ..., h1 = 7, a2 = 8, ..., h8 = 63.
* Classes use FEN piece letters (uppercase white, lowercase black) plus `"empty"`.

Milestone: shared by M2 (auto-labeling), M3 (training targets), M4 (assembly), M5.

**TODO**
- [ ] nothing; do not reorder `CLASSES` once a model has been trained.

In [ ]:
from __future__ import annotations

import chess

## Classes
Order is fixed: model output index `i` means `CLASSES[i]`. The M3 baseline predicts coarser
targets derived from the 13 classes: occupancy (empty / occupied) and color3
(empty / white / black).

In [ ]:
CLASSES: tuple[str, ...] = (
    "empty",
    "P", "N", "B", "R", "Q", "K",   # white
    "p", "n", "b", "r", "q", "k",   # black
)
NUM_CLASSES = len(CLASSES)  # 13
CLASS_TO_INDEX: dict[str, int] = {c: i for i, c in enumerate(CLASSES)}
EMPTY = 0

OCCUPANCY = ("empty", "occupied")
COLOR3 = ("empty", "white", "black")

SQUARE_NAMES: tuple[str, ...] = tuple(chess.SQUARE_NAMES)  # index -> "a1".."h8"

## Conversions

In [ ]:
def label_to_occupancy(label: str) -> int:
    return 0 if label == "empty" else 1


def label_to_color3(label: str) -> int:
    if label == "empty":
        return 0
    return 1 if label.isupper() else 2


def board_to_labels(board: chess.Board) -> list[str]:
    """64 labels in square-index order for a python-chess board."""
    out = []
    for sq in chess.SQUARES:
        piece = board.piece_at(sq)
        out.append("empty" if piece is None else piece.symbol())
    return out


def labels_to_board(labels: list[str]) -> chess.Board:
    """Inverse of board_to_labels. Piece placement only; turn/castling/ep left default."""
    if len(labels) != 64:
        raise ValueError(f"expected 64 labels, got {len(labels)}")
    board = chess.Board(None)
    for sq, lab in enumerate(labels):
        if lab != "empty":
            board.set_piece_at(sq, chess.Piece.from_symbol(lab))
    return board

## Self-check
(was tests/test_labels.py)

In [ ]:
assert NUM_CLASSES == 13 and CLASSES[0] == "empty"

_b = chess.Board()
_labs = board_to_labels(_b)
assert _labs[chess.E1] == "K" and _labs[chess.E8] == "k" and _labs[chess.E4] == "empty"
assert labels_to_board(_labs).board_fen() == _b.board_fen()

assert [label_to_color3(x) for x in ("empty", "Q", "q")] == [0, 1, 2]
assert [label_to_occupancy(x) for x in ("empty", "n")] == [0, 1]
print("labels OK")